### Getting started with text entropy

> The rose that blooms beneath the light of love,
> repeats its bloom each time the light returns;
> the light departs, yet love remains above,
> and love, like light, forever softly burns.

- 4 "documents" (one per line)
- Repeated words: **the**, **light**, **love**, **bloom(s)**


### Build the Frequency Matrix

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

lines = [
    "The rose that blooms beneath the light of love,",
    "repeats its bloom each time the light returns;",
    "the light departs, yet love remains above,",
    "and love, like light, forever softly burns.",
]


# a powerful function from Scikit-learn that moves text into frequencies
vectorizer = CountVectorizer(lowercase=True)

X = vectorizer.fit_transform(lines)

dtm = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"line {i+1}" for i in range(len(lines))]
)

dtm


## Reading the Matrix

- Each **row** = one line (a "document")
- Each **column** = one unique word (the vocabulary)
- Cell values = how many times that word appears in that line
- Column sums give overall word frequency:


In [ ]:
dtm.sum(axis=0).sort_values(ascending=False)


## Word Entropy

Treat the column sums as a probability distribution over the vocabulary,
then compute Shannon entropy $H = -\sum_i p_i \log_2 p_i$:


In [ ]:
import numpy as np

freqs = dtm.sum(axis=0)
probs = freqs / freqs.sum()

entropy = -(probs * np.log2(probs)).sum()


In [ ]:
print(f"Vocabulary size: {len(freqs)}")
print(f"Word entropy H = {entropy:.3f} bits")
print(f"Max possible entropy (uniform) = {np.log2(len(freqs)):.3f} bits")

- Low $H$ → a few words dominate (here, "the", "light", "love" repeat a lot)
- High $H$ → words are spread out more evenly
- Comparing $H$ to the uniform max shows *how skewed* the distribution is


## Entropy per Line

The same idea applied row-wise shows which lines are more "predictable":

In [ ]:
def shannon_entropy(counts):
    '''A simple implementation of Shannon's formula for lists of word frequencies'''
    counts = counts[counts > 0]

    p = counts / counts.sum()

    return -(p * np.log2(p)).sum()




In [ ]:
line_entropy = dtm.apply(shannon_entropy, axis=1)

line_entropy